# dee.cpp Ornith Milestone 3 forensic profile
This notebook is intentionally thin. It pins opt/real-model-t1 HEAD so
all M3 forensic instrumentation commits (02f80c6 persistent staging, 6ed324f
barrier removal, e5a610a device-resident MoE forward) plus the analyzer
instrumentation are present, runs the controlled matrix identically to
Milestone 2.5, and delegates analysis to analyze_milestone3_matrix.py.

In [ ]:
import importlib.metadata, json, os, platform, shutil, subprocess, sys, time
from pathlib import Path
import psutil, torch
print(json.dumps({
    'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda_runtime': torch.version.cuda,
    'cpu_count': os.cpu_count(), 'ram_bytes': psutil.virtual_memory().total,
    'working_disk': shutil.disk_usage('/kaggle/working')._asdict(),
    'gpu_count': torch.cuda.device_count(),
    'gpus': [{'index': i, 'name': torch.cuda.get_device_name(i),
              'memory': torch.cuda.get_device_properties(i).total_memory}
             for i in range(torch.cuda.device_count())],
}, indent=2), flush=True)
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.device_count() == 2, f'dual-T4 assignment required, got {torch.cuda.device_count()} GPUs'
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2))

In [ ]:
import os, json
RUN_ID = os.environ.get('RUN_ID', 'LOCAL_RUN')
COMMIT_EXPECTED = os.environ.get('COMMIT_EXPECTED', '4d8ccf2')
HARNESS_NONCE = os.environ.get('HARNESS_NONCE', 'LOCAL_HARNESS')
print(json.dumps({'RUN_ID': RUN_ID, 'COMMIT_EXPECTED': COMMIT_EXPECTED,
                  'HARNESS_NONCE': HARNESS_NONCE}), flush=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
                'transformers==5.14.1', 'safetensors==0.8.0',
                'pybind11==3.0.1', 'psutil==7.0.0', 'nvidia-ml-py'], check=True)
packages = {name: importlib.metadata.version(name) for name in
            ('transformers', 'safetensors', 'pybind11', 'psutil', 'nvidia-ml-py')}
print(json.dumps({'installed_packages': packages}, indent=2), flush=True)

In [ ]:
# Pin to opt/real-model-t1 HEAD so all post-M3 instrumentation commits
# (including path-proof counters, sync/overlap/multi-gpu helpers in
# run_ornith_forensics.py and the analyze_milestone3_matrix.py script)
# are present.  Checking out the floating branch HEAD (not a hard SHA)
# keeps the verifier authoritative on the latest M3 work.
ROOT = Path('/kaggle/temp/dee-source')
if ROOT.exists():
    assert str(ROOT.resolve()).startswith('/kaggle/temp/')
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--branch', 'opt/real-model-t1', '--single-branch',
                'https://github.com/so-nerdyy/dee.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', 'origin/opt/real-model-t1'], cwd=ROOT, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
DEE = ROOT / 'dee.cpp'
EVIDENCE = Path(f'/kaggle/working/ornith-milestone3-evidence-{RUN_ID}')
EVIDENCE.mkdir(parents=True, exist_ok=True)
print({'repo': str(ROOT), 'commit': commit, 'evidence': str(EVIDENCE)}, flush=True)

In [ ]:
EXPECTED = COMMIT_EXPECTED
_preflight_msg_cell5 = "assert commit == EXPECTED, f'kernel preflight rejected: commit mismatch -- got {commit}, expected {EXPECTED}. The instrumentation target must be re-pinned.'"
_preflight_ok_cell5 = False
try:
    assert commit == EXPECTED, f'kernel preflight rejected: commit mismatch -- got {commit}, expected {EXPECTED}. The instrumentation target must be re-pinned.'
    _preflight_ok_cell5 = True
except BaseException as _preflight_exc:
    import time as _preflight_t, traceback as _preflight_tb
    try:
        with open('/kaggle/working/preflight_failure.txt', 'w', encoding='utf-8') as _pf:
            _pf.write('notebook cell[5] preflight failed\n')
            _pf.write(repr(_preflight_exc) + '\n')
            _pf.write('original assertion: ' + _preflight_msg_cell5 + '\n')
            _pf.write(_preflight_tb.format_exc() + '\n')
    except Exception:
        pass
    print('PREFLIGHT cell[5] FAILED: ' + repr(_preflight_exc), flush=True)
    _preflight_t.sleep(60)
    raise
print(json.dumps({'preflight_1_commit_PASS': True, 'commit': commit}), flush=True)


In [ ]:
candidates = []
for index_path in Path('/kaggle/input').rglob('model.safetensors.index.json'):
    config_path = index_path.parent / 'config.json'
    if config_path.is_file() and json.loads(config_path.read_text()).get('model_type') == 'qwen3_5_moe':
        candidates.append(index_path.parent)
assert len(candidates) == 1, candidates
MODEL = candidates[0]
index = json.loads((MODEL / 'model.safetensors.index.json').read_text())
shards = sorted(set(index['weight_map'].values()))
assert len(shards) == 16 and all((MODEL / name).is_file() for name in shards)
print(json.dumps({'model_dir': str(MODEL), 'tensor_count': len(index['weight_map']),
                  'shard_count': len(shards),
                  'checkpoint_bytes': sum((MODEL / name).stat().st_size for name in shards)},
                 indent=2), flush=True)

In [ ]:
BUILD = DEE / 'build-kaggle-cuda'
_build_started_ns = time.time_ns()
for _stale_so in (DEE / 'pydee').glob('pydee_core*.so'):
    _stale_so.unlink()
subprocess.run(['cmake', '-S', str(DEE), '-B', str(BUILD), '-G', 'Ninja',
                '-DDEE_CUDA=ON', '-DDEE_BUILD_TESTS=ON',
                '-DCMAKE_CUDA_ARCHITECTURES=75', '-DCMAKE_BUILD_TYPE=Release'], check=True)
subprocess.run(['cmake', '--build', str(BUILD), '--parallel', '4'], check=True)
subprocess.run(['ctest', '--test-dir', str(BUILD), '--output-on-failure'], check=True)
subprocess.run([sys.executable, '-m', 'pytest',
                str(DEE / 'tests/test_milestone25_memory.py'),
                str(DEE / 'tests/test_analyze_milestone25_expert_trace.py'),
                str(DEE / 'tests/test_run_ornith_forensics.py'),
                str(DEE / 'tests/test_analyze_milestone25_matrix.py'),
                str(DEE / 'tests/test_analyze_milestone3_matrix.py'),
                str(DEE / 'tests/test_m3_supervisor_v6.py'), '-q'],
               cwd=DEE, check=True)
env = os.environ.copy(); env['DEE_BUILD_DIR'] = str(BUILD)
subprocess.run([sys.executable, str(DEE / 'pydee/setup.py'), 'build_ext', '--inplace', '--force'],
               cwd=DEE, env=env, check=True)
print('CUDA build, native tests, Python tests, and pydee binding passed', flush=True)

In [ ]:
import gc as _gc, hashlib as _hashlib, importlib as _importlib, os as _os
_so_candidates = list((DEE / 'pydee').glob('pydee_core*.so'))
_preflight_msg_cell8 = "assert len(_so_candidates) == 1, 'Preflight #2 rejected: expected exactly one freshly built pydee_core .so'"
_preflight_ok_cell8 = False
try:
    assert len(_so_candidates) == 1, f'Preflight #2 rejected: expected exactly one freshly built pydee_core .so, got {_so_candidates}'
    _preflight_ok_cell8 = True
except BaseException as _preflight_exc:
    import time as _preflight_t, traceback as _preflight_tb
    try:
        with open('/kaggle/working/preflight_failure.txt', 'w', encoding='utf-8') as _pf:
            _pf.write('notebook cell[8] preflight failed\n')
            _pf.write(repr(_preflight_exc) + '\n')
            _pf.write('original assertion: ' + _preflight_msg_cell8 + '\n')
            _pf.write(_preflight_tb.format_exc() + '\n')
    except Exception:
        pass
    print('PREFLIGHT cell[8] FAILED: ' + repr(_preflight_exc), flush=True)
    _preflight_t.sleep(60)
    raise
_so = _so_candidates[0]
assert _so.stat().st_mtime_ns >= _build_started_ns, 'Preflight #2 rejected: pydee_core predates this build invocation'
sys.path.insert(0, str(DEE))
for _module_name in [name for name in sys.modules if name == 'pydee' or name.startswith('pydee.')]:
    del sys.modules[_module_name]
_importlib.invalidate_caches()
import pydee as _pydee
import pydee.pydee_core as _pydee_core
_imported_so = Path(_pydee_core.__file__).resolve()
assert _imported_so == _so.resolve(), f'stale extension import: imported {_imported_so}, built {_so.resolve()}'
_binary = _imported_so.read_bytes()
_markers = [b'[DEE_TA_SELFTEST_BEGIN]', b'[DEE_TA_ALLOC]', b'[DEE_TA_FREE]', b'[DEE_TA_SELFTEST_PASS]']
assert all(marker in _binary for marker in _markers), 'trace self-test markers missing from imported binary'
assert _pydee._trace_alloc_selftest(), 'traced allocation wrapper self-test failed'
_stats_before = dict(_pydee._trace_alloc_stats())
def _make_probe_config(_device_id):
    _cfg = _pydee.EngineConfig()
    _cfg.shard_paths = [str(DEE / 'tests/data/split-router/split-a.safetensors'), str(DEE / 'tests/data/split-router/split-b.safetensors')]
    _cfg.oracle_path = ''
    _cfg.num_tokens = 1; _cfg.num_layers = 6; _cfg.base_layer = 5
    _cfg.hidden = 4; _cfg.inter = 2; _cfg.num_experts = 3; _cfg.topk = 2
    _cfg.device_id = _device_id; _cfg.use_cuda = True
    _cfg.cache_dtype = _pydee.DeviceCacheDType.Fp16
    _cfg.transfer_dtype = _pydee.WeightTransferDType.Bf16
    _cfg.budget_bytes = 2 * 3 * 2 * 4 * 2
    return _cfg
_probe_engine_0 = _pydee.Engine(); _probe_engine_1 = _pydee.Engine()
assert _probe_engine_0.init(_make_probe_config(0)), 'device-0 Engine allocation probe failed'
assert _probe_engine_1.init(_make_probe_config(1)), 'device-1 Engine allocation probe failed'
_stats_real = dict(_pydee._trace_alloc_stats())
assert _stats_real['non_selftest_allocs'] > _stats_before['non_selftest_allocs'], (_stats_before, _stats_real)
torch.cuda.set_device(0)
del _probe_engine_1
_gc.collect(); torch.cuda.synchronize(1)
del _probe_engine_0
_gc.collect(); torch.cuda.synchronize(0)
_stats_after = dict(_pydee._trace_alloc_stats())
assert _stats_after['live'] == _stats_before['live'], (_stats_before, _stats_after)
assert all(_stats_after[name] == 0 for name in ('unalloc_aborts', 'double_free_aborts', 'mismatch_aborts', 'uaf_aborts')), _stats_after
_identity = {'schema_version': 1, 'run_id': RUN_ID, 'commit': commit,
             'harness_nonce': HARNESS_NONCE,
             'pydee_core_path': str(_imported_so),
             'pydee_core_sha256': _hashlib.sha256(_binary).hexdigest(),
             'marker_strings_present': [marker.decode() for marker in _markers],
             'trace_stats_before_real_probe': _stats_before,
             'trace_stats_during_real_probe': _stats_real,
             'trace_stats_after_probe_teardown': _stats_after}
(EVIDENCE / 'commit-binary-identity.json').write_text(json.dumps(_identity, indent=2) + '\n')
_build_manifest = {'schema_version': 1, 'run_id': RUN_ID, 'commit': commit,
                   'harness_nonce': HARNESS_NONCE,
                   'build_started_ns': _build_started_ns,
                   'binary_mtime_ns': _imported_so.stat().st_mtime_ns,
                   'cmake_cache': str(BUILD / 'CMakeCache.txt'),
                   'cuda_architectures': '75', 'extension_force_rebuilt': True}
(EVIDENCE / 'build-manifest.json').write_text(json.dumps(_build_manifest, indent=2) + '\n')
print(json.dumps({'preflight_2_build_import_trace_PASS': True, 'identity': _identity}, indent=2), flush=True)


In [ ]:
def run_tee(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=DEE, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

# Run the same seven variants entered by M3 v4, with crash-safe progress
# records written by run_milestone25_matrix.py after every transition.
matrix_command = [sys.executable, '-u', '-X', 'faulthandler',
                  str(DEE / 'scripts/run_milestone25_matrix.py'),
                  '--model-dir', str(MODEL), '--output-dir', str(EVIDENCE),
                  '--require-dual-gpu', '--skip-aggregate', '--run-ids',
                  'dual-cold-primary', 'dual-warm-profiled',
                  'dual-warm-control', 'dual-warm-reference-present',
                  'dual-cache-disabled', 'dual-cache-capacity-4',
                  'dual-long-prompt',
                  '--kernel-slug', 'nivind/dee-cpp-ornith-milestone-3-forensics']
run_tee(matrix_command, EVIDENCE / 'logs/matrix-driver.log')
print('Controlled matrix finished; cold run was first', flush=True)

In [ ]:
router_report = EVIDENCE / 'ornith-router-parity.json'
subprocess.run([sys.executable, '-u', '-X', 'faulthandler',
                str(DEE / 'scripts/run_ornith_router_parity.py'),
                '--model-dir', str(MODEL), '--layers', '0', '3', '20', '39',
                '--tokens', '16', '--report', str(router_report)], cwd=DEE, check=True)
router = json.loads(router_report.read_text()); assert router['result'] == 'PASS'
assert all(item['expert_ids_exact'] for item in router['layers'])
layer0_report = EVIDENCE / 'ornith-layer0-regression.json'
subprocess.run([sys.executable, '-u', '-X', 'faulthandler',
                str(DEE / 'scripts/run_ornith_layer0_parity.py'),
                '--model-dir', str(MODEL), '--max-prompt-tokens', '4',
                '--report', str(layer0_report)], cwd=DEE, check=True)
layer0 = json.loads(layer0_report.read_text()); assert layer0['pass'] is True
print({'router': router['result'], 'layer0': layer0['pass']}, flush=True)

In [ ]:
# The M2.5 analyzer would compare rounds to M2.5 baseline; for M3 we
# invoke the dedicated M3 comparator that reads path-proof.json, sync,
# overlap, multi-gpu timeline and emits MILESTONE_3_VERIFICATION.md
# plus the documented M3 deliverable list.
analysis_command = [sys.executable, '-u', str(DEE / 'scripts/analyze_milestone3_matrix.py'),
                    '--m3-dir', str(EVIDENCE),
                    '--m25-dir', str(DEE / 'benchmark_reports/milestone-2.5/kaggle-forensics-latest-output/ornith-milestone25-evidence'),
                    '--output-dir', str(EVIDENCE / 'analysis')]
run_tee(analysis_command, EVIDENCE / 'logs/final-analysis.log')
final_report = json.loads((EVIDENCE / 'analysis/milestone-3-report.json').read_text())
assert final_report['result'] == 'PASS', final_report['result']
summary = final_report['defects_summary']
failures = [label for label, count in summary.items() if label in ('inconclusive', 'regressed') and count > 0]
assert not failures, {'forensic_result': final_report['result'], 'non_compliant': failures}
print({'forensic_result': final_report['result'], 'defects_summary': summary}, flush=True)

In [ ]:
_root_required = ['matrix-summary.json', 'final_report.json', 'summary_metrics.csv',
                  'raw-allocation-trace.log', 'build-manifest.json',
                  'commit-binary-identity.json',
                  'ornith-router-parity.json',
                  'ornith-layer0-regression.json']
_analysis_required = ['MILESTONE_3_VERIFICATION.md', 'milestone-3-report.json',
                      'before-after-milestone25.json', 'acceptance-audit.json',
                      'correctness-report.json', 'environment.json',
                      'matrix-summary.json', 'memory-timeline.json',
                      'host-memory-breakdown.json', 'gpu-memory-breakdown.json',
                      'layer-timing.json', 'transfer-analysis.json',
                      'expert-cache-analysis.json', 'synchronization-analysis.json',
                      'overlap-analysis.json', 'multi-gpu-timeline.json',
                      'path-proof.json', 'profiler-summary.md',
                      'bottleneck-ranking.json', 'expert-trace.jsonl.gz',
                      'evidence-integrity-sha256.txt']
_required_paths = [EVIDENCE / name for name in _root_required]
_required_paths += [EVIDENCE / 'analysis' / name for name in _analysis_required]
_expected_run_ids = ['dual-cold-primary', 'dual-warm-profiled',
        'dual-warm-control', 'dual-warm-reference-present',
        'dual-cache-disabled', 'dual-cache-capacity-4', 'dual-long-prompt']
_per_run_core = ['run-report.json', 'memory-timeline.json', 'layer-timing.json',
                 'timing-raw.json', 'gpu-utilization-summary.json',
                 'synchronization-analysis.json', 'overlap-analysis.json',
                 'multi-gpu-timeline.json', 'path-proof.json',
                 'expert-trace.jsonl']
for _run_id in _expected_run_ids:
    _required_paths += [EVIDENCE / 'runs' / _run_id / name for name in _per_run_core]
    if _run_id != 'dual-warm-control':
        _required_paths += [EVIDENCE / 'runs' / _run_id / 'expert-cache-analysis.json',
                            EVIDENCE / 'runs' / _run_id / 'transfer-analysis.json']
_empty_allowed = EVIDENCE / 'runs' / 'dual-warm-control' / 'expert-trace.jsonl'
_missing = [str(path.relative_to(EVIDENCE)) for path in _required_paths
            if not path.is_file() or (path.stat().st_size == 0 and path != _empty_allowed)]
assert not _missing, f'missing or empty required files: {_missing}'
_matrix = json.loads((EVIDENCE / 'matrix-summary.json').read_text())
assert _matrix.get('selected_run_ids') == _expected_run_ids, _matrix
assert len(_matrix['experiments']) == 7 and all(item.get('result') == 'PASS' for item in _matrix['experiments']), _matrix
for _run_id in _expected_run_ids:
    _run_report = json.loads((EVIDENCE / 'runs' / _run_id / 'run-report.json').read_text())
    assert _run_report.get('run_id') == _run_id, (_run_id, _run_report.get('run_id'))
    assert _run_report.get('git_commit') == commit, (_run_id, _run_report.get('git_commit'), commit)
    assert _run_report.get('result') == 'PASS', (_run_id, _run_report.get('result'))
assert '[ta_alloc]' in (EVIDENCE / 'raw-allocation-trace.log').read_text(errors='replace'), 'no real traced dee.cpp allocation captured'
_artifact_entries = []
for _artifact in sorted(path for path in EVIDENCE.rglob('*') if path.is_file() and path.name != 'artifact-manifest.json'):
    _artifact_entries.append({'path': _artifact.relative_to(EVIDENCE).as_posix(),
                              'bytes': _artifact.stat().st_size,
                              'sha256': _hashlib.sha256(_artifact.read_bytes()).hexdigest()})
_artifact_manifest = {'schema_version': 1, 'result': 'PASS', 'run_id': RUN_ID,
                      'harness_nonce': HARNESS_NONCE,
                      'commit': commit, 'required_paths': [str(path.relative_to(EVIDENCE)) for path in _required_paths],
                      'seven_pass_matrix': True, 'artifacts': _artifact_entries}
(EVIDENCE / 'artifact-manifest.json').write_text(json.dumps(_artifact_manifest, indent=2) + '\n')
archive = shutil.make_archive('/kaggle/working/ornith-milestone3-evidence', 'gztar',
                              root_dir=EVIDENCE.parent, base_dir=EVIDENCE.name)
print(json.dumps({'final_status': 'PASS', 'archive': archive,
                  'run_id': RUN_ID, 'artifact_count': len(_artifact_entries)},
                 indent=2), flush=True)